# Hopfield 统一任务基准：Phase 1

状态：`implementation-only / not-run`

本 notebook 验证一件事：在数据、线索、随机种子、停止条件和指标完全相同的情况下，只替换模型存储与检索组件，能否把不同 Hopfield 模型作为同一张图上的曲线进行比较。当前接入 Classical Hopfield 与三次 Polynomial DAM。

## 0. 实验管线

`a 记忆输入 → b 模型存储 → c 检索线索 → d 检索动力学 → e 测量 → f 同图比较`

其中 `a/c/e/f` 对所有模型共用；当前只替换 `b/d`。Notebook 不安装额外依赖，直接使用 Colab 自带的 PyTorch、pandas 与 Matplotlib。

In [ ]:
from pathlib import Path
import subprocess
import sys

source_candidates = [
    Path.cwd() / "src",
    Path.cwd() / "hopfield-benchmark" / "src",
    Path("/content/nn-labs/hopfield-benchmark/src"),
]
source_root = next((item for item in source_candidates if item.exists()), None)
if source_root is None:
    checkout = Path("/content/nn-labs")
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/Heptazero/nn-labs.git",
            str(checkout),
        ],
        check=True,
    )
    source_root = checkout / "hopfield-benchmark" / "src"
sys.path.insert(0, str(source_root))
repo_root = source_root.parents[1]
git_commit = subprocess.check_output(
    ["git", "-C", str(repo_root), "rev-parse", "HEAD"],
    text=True,
).strip()
print(f"benchmark source: {source_root}")
print(f"git commit: {git_commit}")

In [ ]:
from dataclasses import asdict
from datetime import datetime, timezone
import json
from uuid import uuid4

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, display

from hopfield_benchmark import (
    BenchmarkConfig,
    ClassicalHopfield,
    PolynomialDAM,
    a1_make_independent_binary,
    c1_make_hamming_cue,
    capacity_summary,
    f1_plot_recall_vs_corruption,
    f2_plot_recall_vs_load,
    f3_plot_capacity_scaling,
    f4_plot_quality_cost,
    f5_plot_error_dynamics,
    run_paired_benchmark,
    validate_paired_results,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 40)
print(f"torch={torch.__version__}, pandas={pd.__version__}")

## 1. b/d：模型注册表

注册表只保存构造函数。运行器会先生成一个共享案例，再分别实例化和拟合模型。`model_id` 是曲线颜色的唯一来源。

In [ ]:
MODEL_FACTORIES = {
    "classical_hebb": ClassicalHopfield,
    "polynomial_dam_d3": lambda: PolynomialDAM(degree=3),
}
MODEL_FACTORIES

## 2. 公共组件自检

这里不是实验结论，只检查接口是否闭合：两个模型接收同一个 memory tensor 和 cue tensor；异步更新结束后返回统一字段；各自定义的能量在记录精度内不增加。

In [ ]:
check_memories = a1_make_independent_binary(N=32, P=4, data_seed=7)
check_target = check_memories.patterns[0]
check_cue = c1_make_hamming_cue(check_target, corruption_level=0.125, cue_seed=11)

for registered_id, factory in MODEL_FACTORIES.items():
    check_model = factory().fit(check_memories.patterns)
    assert check_model.model_id == registered_id
    check_result = check_model.retrieve(
        check_cue,
        target=check_target,
        update_seed=13,
        max_sweeps=8,
    )
    energy_deltas = torch.diff(torch.tensor(check_result.energy_trace))
    assert bool(torch.all(energy_deltas <= 1e-10))
    assert check_result.status in {"fixed", "max_steps"}
    print(registered_id, check_result.status, check_result.sweeps)

print("component contract: passed")

## 3. Gate 1 有限扫描配置

这个扫描有明确上限，不自动扩大。它用于检查基线和图形接口，不足以估计渐近容量指数。两个模型共享所有 case seeds，且当前只比较 `native / equal_max_sweeps`；严格的同存储和同 FLOPs 比较留到下一道门。

In [ ]:
execution_id = (
    datetime.now(timezone.utc).strftime("phase1-%Y%m%dT%H%M%SZ-")
    + uuid4().hex[:8]
)
CONFIG = BenchmarkConfig(
    N_values=(64, 128),
    P_values=(4, 8, 16, 32),
    corruption_levels=(0.0, 0.1, 0.2, 0.3),
    pattern_sets=3,
    targets_per_set=4,
    max_sweeps=20,
    base_seed=20260905,
    experiment_id=execution_id,
    git_commit=git_commit,
)
display(pd.Series(asdict(CONFIG), name="value").to_frame())

## 4. 运行配对案例并保存原始记录

每一行是一条目标记忆的一次检索。失败和达到 `max_sweeps` 的记录不会被删除。输出使用 JSON Lines，列表轨迹不会在 CSV 中被静默改形。

In [ ]:
results = run_paired_benchmark(MODEL_FACTORIES, CONFIG)
validate_paired_results(results, expected_models=tuple(MODEL_FACTORIES))

artifact_root = (
    Path("/content/hopfield-benchmark-results")
    if Path("/content").exists()
    else Path.cwd() / "hopfield-benchmark-results"
)
artifact_root.mkdir(parents=True, exist_ok=True)
results.to_json(
    artifact_root / "phase1_raw_results.jsonl",
    orient="records",
    lines=True,
)
with (artifact_root / "phase1_config.json").open("w", encoding="utf-8") as handle:
    json.dump(asdict(CONFIG), handle, ensure_ascii=False, indent=2)
print(f"saved {len(results)} raw rows to {artifact_root}")

In [ ]:
display(
    results.drop(columns=["energy_trace", "error_trace"])
    .sort_values(["run_id", "model_id"])
    .head(8)
)
display(results.groupby(["model_id", "status"]).size().rename("count").to_frame())

## 实验 1：U2 噪声恢复——同一张图直接比较模型

固定 `N=128, P=16`，横轴是相同 Hamming 损坏比例，纵轴是 exact recall。每一个 trial 在作图前都必须同时包含两种模型，否则函数拒绝画图。

In [ ]:
f1_plot_recall_vs_corruption(results, N=128, P=16)
plt.show()

In [ ]:
u2_endpoint = (
    results[(results["N"] == 128) & (results["P"] == 16) & (results["corruption_level"] == 0.3)]
    .groupby("model_id")["exact_recall"]
    .agg(["mean", "count"])
)
u2_lines = [
    f"- `{model_id}`：rho=0.30 时 exact recall={row['mean']:.3f}，n={int(row['count'])}。"
    for model_id, row in u2_endpoint.iterrows()
]
display(Markdown(
    "**本次运行的描述性结论**\n\n"
    + "\n".join(u2_lines)
    + "\n\n**证据边界**：这里只是有限规模、native budget 的配对结果。阴影是正态近似区间；样本较少时不能据此宣称容量阶数或统计优势。"
))

## 实验 2：U3 有限负载曲线

固定 `N=128` 和 10% Hamming 损坏。横轴使用实际存储数 `P`，而不是把不同容量增长阶硬塞进同一个 `P/N`。

In [ ]:
f2_plot_recall_vs_load(results, N=128, corruption_level=0.1)
plt.show()

In [ ]:
u3_rates = (
    results[(results["N"] == 128) & (results["corruption_level"] == 0.1)]
    .groupby(["model_id", "P"])["exact_recall"]
    .mean()
    .unstack("P")
)
display(Markdown(
    "**本次运行的描述性结论**：下表是上图对应的条件均值，可直接检查曲线差异是否来自同一个 `P`。\n\n"
    "**证据边界**：曲线可能非单调；预设扫描点之外不插值，也不把最后一个成功点自动称为理论容量。"
))
display(u3_rates.round(3))

## 实验 3：U3 容量缩放与右删失

临界容量使用预先固定的 90% clean one-sweep-unchanged 判据：从原记忆开始完成一轮异步更新后，状态必须完全不变。向上空心点表示右删失，容量至少达到扫描上界；向下空心点表示左删失，容量低于或等于扫描下界。

In [ ]:
capacities = capacity_summary(results, success_threshold=0.9)
capacities.to_csv(artifact_root / "phase1_capacity_summary.csv", index=False)
display(capacities)
f3_plot_capacity_scaling(capacities)
plt.show()

In [ ]:
capacity_lines = []
for row in capacities.itertuples(index=False):
    relation = "≤" if row.left_censored else "≥" if row.right_censored else "="
    capacity_lines.append(f"- `{row.model_id}`，N={row.N}：P_c {relation} {row.P_c}。")
display(Markdown(
    "**本次运行的描述性结论**\n\n"
    + "\n".join(capacity_lines)
    + "\n\n**证据边界**：这是当前离散扫描网格上的经验阈值。右删失点不能参与普通最小二乘拟合，也不能用于声称线性、多项式或指数容量。"
))

## 实验 4：U5 配对动力学

选择同一个 `run_id`，叠加两种模型的 bit error 轨迹。模型能量定义不同，因此主图不把两种能量数值混在一个纵轴上。

In [ ]:
dynamics_run_id = results[
    (results["N"] == 128)
    & (results["P"] == 16)
    & (results["corruption_level"] == 0.2)
]["run_id"].iloc[0]
f5_plot_error_dynamics(results, run_id=dynamics_run_id)
plt.show()

In [ ]:
dynamics_rows = results[results["run_id"] == dynamics_run_id]
dynamics_lines = []
for row in dynamics_rows.itertuples(index=False):
    final_error = row.error_trace[-1] if row.error_trace else float("nan")
    dynamics_lines.append(
        f"- `{row.model_id}`：status={row.status}，sweeps={row.sweeps}，final bit error={final_error:.3f}。"
    )
display(Markdown(
    "**本次运行的描述性结论**\n\n"
    + "\n".join(dynamics_lines)
    + "\n\n**证据边界**：单条轨迹只解释更新过程，不代表总体召回率。总体判断必须回到前面的配对重复实验。"
))

## 实验 5：U7 质量—计算量视图

横轴是 adapter 中登记的近似操作数，不是 GPU/CPU 实测延迟。它用于暴露 native 曲线的资源差异，而不是完成严格的 matched-compute 比较。

In [ ]:
f4_plot_quality_cost(results, N=128, corruption_level=0.1)
plt.show()

In [ ]:
resource_table = (
    results[(results["N"] == 128) & (results["corruption_level"] == 0.1)]
    .groupby("model_id", as_index=False)
    .agg(
        exact_recall=("exact_recall", "mean"),
        retrieval_flops=("retrieval_flops", "mean"),
        storage_bytes=("storage_bytes", "mean"),
    )
)
display(Markdown(
    "**本次运行的描述性结论**：同图中的质量差异必须与下面的存储和操作数一起读。\n\n"
    "**证据边界**：当前不做硬件速度结论；不同实现的向量化程度会改变 wall-clock time。"
))
display(resource_table.round(3))

## 5. 下一道实验门

只有在 Colab 输出确认 Classical Hopfield 的干净固定点、异步能量不增和有限规模容量基线合理，并确认 Polynomial DAM 的单坐标能量不增与 degree=2 退化关系后，才进入严格的 `matched_storage`、`matched_compute` 以及更多模型。

本 notebook 的图可以直接比较模型，但“能画在同一张图上”只说明任务配对成立，不自动说明预算公平，也不自动支持论文级容量结论。